# Twitter Sentiment Classification: Positive vs. Negative

In [ ]:
!pip install pandas
!pip install ekphrasis

!pip install -q scikit-learn
!pip install -q gensim
!pip install numpy

!pip install textblob

!pip install emoji


import pandas as pd

df_train = pd.read_csv('twitter_sentiment_train.csv')
df_test  = pd.read_csv('twitter_sentiment_test.csv')

In [ ]:
int_to_label = {1: 'Positive', 0: 'Negative'}

In [ ]:
df_train.head(2)

In [ ]:
# Using ekphrasis for tokenization

from ekphrasis.classes.preprocessor import TextPreProcessor
from ekphrasis.classes.tokenizer import SocialTokenizer
from ekphrasis.dicts.emoticons import emoticons

text_processor = TextPreProcessor(
    normalize=['url', 'email', 'percent', 'money', 'phone', 'user',
               'time', 'date', 'number'],
    annotate={"hashtag", "allcaps","elongated"},
    fix_html=True,
    segmenter="twitter",
    corrector="twitter",
    unpack_hashtags=True,
    unpack_contractions=True,
    tokenizer=SocialTokenizer(lowercase=True).tokenize,

    dicts = [emoticons]
)

# Test tokenization
tweet = "Amazing movie!! 😂🔥 Check it out at https://t.co/example @user #BestFilm"
tokens = text_processor.pre_process_doc(tweet)
print(tokens)

In [ ]:
# Tokenize our testing and training datasets using twokenize
df_train["tokens"] = df_train["text"].apply(text_processor.pre_process_doc)
df_test["tokens"] = df_test["text"].apply(text_processor.pre_process_doc)

print(df_train.head(2))
print(df_test.head(2))

In [ ]:
# Feature extraction ----------
# Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(
        analyzer = lambda x: x, # Tells the vectorizer that the doc is already tokenized
        ngram_range  = (1,1),
        min_df       = 2,
        max_df       = 0.9
        )

bow_vectorizer.fit(df_train["tokens"])

X_train_bow = bow_vectorizer.transform(df_train["tokens"])
X_test_bow = bow_vectorizer.transform(df_test["tokens"])

# Print the shapes to check
print("Train Matrix", X_train_bow.shape)
print("Train Matrix", X_test_bow.shape)



In [ ]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    analyzer = lambda x: x, #Exact same settings as BoW
    ngram_range  = (1, 1),
    min_df       = 2,
    max_df       = 0.9
)

tfidf_vectorizer.fit(df_train["tokens"])

X_train_tfidf = tfidf_vectorizer.transform(df_train["tokens"])
X_test_tfidf = tfidf_vectorizer.transform(df_test["tokens"])

print("Train Matrix", X_train_tfidf.shape)
print("Train Matrix", X_test_tfidf.shape)

In [ ]:
# word2vec

from gensim.models import Word2Vec
import numpy

w2v_model = Word2Vec(
    sentences=df_train["tokens"],
    vector_size=100,
    window=5,
    min_count=1,
)

print("Vocabulary:", list(w2v_model.wv.key_to_index.keys()))
print("Vocabulary size:", len(list(w2v_model.wv.key_to_index.keys())))

# we convert the tweets to vectors using our model
def get_tweet_vectors_w2v(tokens, w2v_model):
  tweet_vectors = []

  for token in tokens:
    if token in w2v_model.wv:
      tweet_vectors.append(w2v_model.wv[token])

  if(len(tweet_vectors) == 0):
    return numpy.mean(100)

  return numpy.mean(tweet_vectors, axis=0)

X_train_w2v = [get_tweet_vectors_w2v(token_line ,w2v_model) for token_line in df_train["tokens"]]
X_test_w2v = [get_tweet_vectors_w2v(token_line ,w2v_model) for token_line in df_test["tokens"]]

print(X_train_w2v[0])
print(X_test_w2v[0])

In [ ]:
# Text Blob
# Extract sentiment features (Poalrity and Subjectivity)

from textblob import TextBlob

def get_polarity_subjectivity(tweet_text):
  blob = TextBlob(tweet_text)
  polarity = blob.sentiment.polarity
  subjectivity = blob.sentiment.subjectivity

  return polarity, subjectivity

#find sentiment features for both of our dataframes
train_polarity = []
train_subjectivity = []
for text in df_train["text"]:
  polarity, subjectivity = get_polarity_subjectivity(text)
  train_polarity.append(polarity)
  train_subjectivity.append(subjectivity)

test_polarity = []
test_subjectivity = []
for text in df_test["text"]:
  polarity, subjectivity = get_polarity_subjectivity(text)
  test_polarity.append(polarity)
  test_subjectivity.append(subjectivity)

print("Train Polarity: " + str(train_polarity))
print("Train Subjectivity: " + str(train_subjectivity))
print("Test Polarity: " + str(test_polarity))
print("Test Subjectivity: " + str(test_subjectivity))

In [ ]:
# Lexicon-Based Counts #TODO
import nltk
from nltk.corpus import opinion_lexicon

nltk.download('opinion_lexicon')

positive_words = set(opinion_lexicon.positive())
negative_words = set(opinion_lexicon.negative())

# print(positive_words)
# print(negative_words)

def count_lexicon(tokens):
  count_positive = 0
  count_negative = 0

  for token in tokens:
    if token in positive_words:
      count_positive+=1
    if token in negative_words:
      count_negative+=1
  return count_positive, count_negative

train_count_positive= []
train_count_negative= []

for tokens in df_train["tokens"]:
  count_p, count_n = count_lexicon(tokens)
  train_count_positive.append(count_p)
  train_count_negative.append(count_n)

test_count_positive= []
test_count_negative= []

for tokens in df_test["tokens"]:
  count_p, count_n = count_lexicon(tokens)
  test_count_positive.append(count_p)
  test_count_negative.append(count_n)

print(train_count_positive[:5])
print(train_count_negative[:5])
print(test_count_positive[:5])
print(test_count_negative[:5])


In [ ]:
# Number of @s

import re

def count_mentions(text):
  return len(re.findall(r'@\w+', text))

train_count_mentions = []
test_count_mentions = []

for text in df_train['text']:
  train_count_mentions.append(count_mentions(text))
for text in df_test['text']:
  test_count_mentions.append(count_mentions(text))

print(train_count_mentions[:10])
print(test_count_mentions[:10])

In [ ]:
# Number of URLS

def count_urls(text):
  return len(re.findall(r'https?://[^\s]+', text))

train_count_urls= []
test_count_urls = []

for text in df_train['text']:
  train_count_urls.append(count_urls(text))
for text in df_test['text']:
  test_count_urls.append(count_urls(text))


print(train_count_urls[:20])
print(test_count_urls[:20])

In [ ]:
# Number of Hastags
def count_hashtags(text):
  return len(re.findall(r'#\w+', text))

train_count_hashtags= []
test_count_hashtags = []

for text in df_train['text']:
  train_count_hashtags.append(count_hashtags(text))
for text in df_test['text']:
  test_count_hashtags.append(count_hashtags(text))


print(train_count_hashtags[:20])
print(test_count_hashtags[:20])

In [ ]:
# Profanity Words
with open('profanity.txt', 'r') as f: profanity_words = f.readlines()
profanity_words = [s.strip() for s in profanity_words]

def count_profanity(tokens):
  result = 0
  for token in tokens:
    if token in profanity_words:
      result+=1
  return result

train_count_profanity= []
test_count_profanity = []

for tokens in df_train['tokens']:
  train_count_profanity.append(count_profanity(tokens))
for tokens in df_test['tokens']:
  test_count_profanity.append(count_profanity(tokens))

print(train_count_profanity[:20])
print(test_count_profanity[:20])


In [ ]:
# Tweet Length

train_tweet_length= []
test_tweet_length = []

for text in df_train['text']:
  train_tweet_length.append(len(text))
for text in df_test['text']:
  test_tweet_length.append(len(text))

print(train_tweet_length[:20])
print(test_tweet_length[:20])

In [ ]:
# Number of Emojis
import emoji

def count_emojis(tokens):
  count =0
  for token in tokens:
    if emoji.is_emoji(token):
      count+=1
  return count

train_count_emojis= []
test_count_emojis = []

for tokens in df_train['tokens']:
  train_count_emojis.append(count_emojis(tokens))
for tokens in df_test['tokens']:
  test_count_emojis.append(count_emojis(tokens))

print(train_count_emojis[:20])
print(test_count_emojis[:20])

In [ ]:
# Capitalization Ratio

def capitalization_ratio(text):
  total_l = 0
  capital_l=0

  for char in text:
    if char.isalpha():
      total_l+=1
      if char.isupper():
        capital_l+=1
  if total_l == 0:
    return 0.0
  else:
    return capital_l/total_l

train_capital_ratio= []
test_capital_ratio = []

for text in df_train['text']:
  train_capital_ratio.append(capitalization_ratio(text))
for text in df_test['text']:
  test_capital_ratio.append(capitalization_ratio(text))

print(train_capital_ratio[:20])
print(test_capital_ratio[:20])


In [ ]:
# Number of exclamation and question marks

def capitalization_exclamation_question(text):
  countex=0
  countq=0

  for char in text:
    if char == '!':
      countex+=1
    if char == '?':
      countq+=1
  return countex,countq

train_ex_count= []
train_q_count= []
test_ex_count =[]
test_q_count=[]

for text in df_train['text']:
  ex, q = capitalization_exclamation_question(text)
  train_ex_count.append(ex)
  train_q_count.append(q)
for text in df_test['text']:
  ex, q = capitalization_exclamation_question(text)
  test_ex_count.append(ex)
  test_q_count.append(q)

print(train_ex_count[:20])
print(train_q_count[:20])
print(test_ex_count[:20])
print(test_q_count[:20])

In [ ]:
# Number of Negation words
negation_words = {"no","not","never","none","nothing","neither","hardly","scarcely",
                  "doesn't","isn't","shouldn't","wouldn't","couldn't","won't","can't",
                  "don't","didn't","hasn't","hadn't","ain't"}

def count_negations(tokens):
  count =0
  for token in tokens:
    if token in negation_words:
      count+=1
  return count

train_count_negation_words= []
test_count_negation_words = []

for tokens in df_train['tokens']:
  train_count_negation_words.append(count_negations(tokens))
for tokens in df_test['tokens']:
  test_count_negation_words.append(count_negations(tokens))


print(train_count_negation_words[:20])
print(test_count_negation_words[:20])

In [ ]:
####### ADDITIONAL FEATURES ##########

In [ ]:
# Get Elongated Words
def getElongatedWords(tokens):
    count = 0
    for i in tokens:
        if(i == "<elongated>"):
            count = count + 1
    return count

train_elongatedNumber = []
test_elongatedNumber = []

for tokens in df_train['tokens']:
  train_elongatedNumber.append(getElongatedWords(tokens))
for tokens in df_test['tokens']:
  test_elongatedNumber.append(getElongatedWords(tokens))

print(train_elongatedNumber[:20])
print(test_elongatedNumber[:20])

In [ ]:
##### MODEL TRAINING #######

In [ ]:
from scipy.sparse import hstack, csr_matrix

# Combine all of our features
X_train_numeric = numpy.column_stack([
  train_polarity,
  train_subjectivity,
  train_count_positive,
  train_count_negative,
  train_count_mentions,
  train_count_urls,
  train_count_hashtags,
  train_count_profanity,
  train_tweet_length,
  train_count_emojis,
  train_capital_ratio,
  train_ex_count,
  train_q_count,
  train_count_negation_words,
  train_elongatedNumber
])

X_test_numeric = numpy.column_stack([
  test_polarity,
  test_subjectivity,
  test_count_positive,
  test_count_negative,
  test_count_mentions,
  test_count_urls,
  test_count_hashtags,
  test_count_profanity,
  test_tweet_length,
  test_count_emojis,
  test_capital_ratio,
  test_ex_count,
  test_q_count,
  test_count_negation_words,
  test_elongatedNumber
])

X_train_numeric = csr_matrix(X_train_numeric)
X_test_numeric = csr_matrix(X_test_numeric)




In [ ]:
# Convert the w2v Lexical features so we can include them in the column stack

# Averages all of the vectors to a set size (100)
# In case of a token not existing, it returns a 0 array
def get_w2v_vectors(tokens, w2v_model):
  vectors =[]
  for token in tokens:
    if token in w2v_model.wv:
      vectors.append(w2v_model.wv[token])
  if len(vectors) == 0:
    return numpy.zeros(100)
  else:
    return numpy.mean(vectors, axis=0)

# Convert all the vectors to a numpy array
X_train_w2v = numpy.array([get_w2v_vectors(tokens, w2v_model) for tokens in df_train["tokens"]])
X_test_w2v = numpy.array([get_w2v_vectors(tokens, w2v_model) for tokens in df_test["tokens"]])
X_train_w2v = csr_matrix(X_train_w2v) # Convert to sparce matrix
X_test_w2v = csr_matrix(X_test_w2v)

In [ ]:
# Create the final training and testing sets
X_train_final = hstack([X_train_numeric, X_train_w2v, X_train_bow])
X_test_final = hstack([X_test_numeric, X_test_w2v, X_test_bow])

print(X_train_final.shape)
print(X_test_final.shape)

In [ ]:
y_train = df_train["label"].values
y_test = df_test["label"].values

from sklearn.linear_model import LogisticRegression

# Evaluate the model
clf = LogisticRegression(max_iter=10000)
clf.fit(X_train_final, y_train)

y_pred = clf.predict(X_test_final)

from sklearn.metrics import accuracy_score, classification_report

# Accuracy of the model trained on ALL the features
print("accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
## APPLY FEATURE SELECTION
from sklearn.feature_selection import SelectFromModel

# Use Linear Regression with L1 penalty, in order to discard unimportant features
selector = SelectFromModel(
    LogisticRegression(penalty='l1', solver='liblinear', max_iter=5000)
)
selector.fit(X_train_final, y_train)

selected_indices = selector.get_support(indices=True)
print("Selected features: ", selected_indices)

# Apply the transformation to our data
X_train_selected = selector.transform(X_train_final)
X_test_selected = selector.transform(X_test_final)
print(X_train_selected.shape, X_test_selected.shape)


In [ ]:
# And create a new classifier on the selected features
clf_selected = LogisticRegression(max_iter=10000)

# Cross Validation
from sklearn.model_selection import cross_val_score
# Perforn 5-fold cross validation
cv_scores = cross_val_score(clf_selected, X_train_selected, y_train, cv=5, scoring='accuracy')

print("Cross Validation Scores:", cv_scores)



In [ ]:
# Train the new classifier of selected features
clf_selected.fit(X_train_selected, y_train)
y_pred_selected = clf_selected.predict(X_test_selected)

# Accuracy of the model after applying Feature Selection
print("New Accuracy: ", accuracy_score(y_test, y_pred_selected))
print(classification_report(y_test, y_pred_selected))


In [ ]:
import pickle

# Save the model
with open('model.pkl', 'wb') as f:
  pickle.dump(clf_selected, f)
# Load the model
with open('model.pkl', 'rb') as f:
  clf_selected = pickle.load(f)
